In [27]:
import pandas as pd
import numpy as np
train_data=pd.read_csv('train.csv')
test_data=pd.read_csv('test.csv')
print(train_data.head(10))
train_data.info()

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   
5            6         0       3   
6            7         0       1   
7            8         0       3   
8            9         1       3   
9           10         1       2   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   
5                                   Moran, Mr. James    male   NaN      0   
6                            McCarthy, Mr. Timothy J    male  54

In [28]:
%pwd

'C:\\Users\\SHUS\\PY'

In [29]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\SHUS\PY
['.ipynb_checkpoints', "code predicting tatanic's problem based on logistic regression.ipynb", "code predicting tatanic's problem based on random forest classifier.ipynb", 'gender_submission.csv', 'office.log', 'project.log', 'submission.csv', 'test.csv', 'train.csv', 'Untitled.ipynb', 'Untitled1.ipynb', 'Untitled10.ipynb', 'Untitled11.ipynb', 'Untitled12.ipynb', 'Untitled13.ipynb', 'Untitled14.ipynb', 'Untitled15.ipynb', 'Untitled16.ipynb', 'Untitled17.ipynb', 'Untitled18.ipynb', 'Untitled2.ipynb', 'Untitled3.ipynb', 'Untitled4.ipynb', 'Untitled5.ipynb', 'Untitled6.ipynb', 'Untitled7.ipynb', 'Untitled8.ipynb', 'Untitled9.ipynb']


In [30]:
train_data['title']=train_data['Name'].str.extract(r'([A-Za-z]+)\.',expand=False)
print(train_data['title'].value_counts())

title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64


In [31]:
#处理各种称呼
train_data['title']=train_data['title'].replace(['Countess','Capt','Col','Don','Dr','Major','Rev','Jonkheer','Dona'],'rare')
train_data['title']=train_data['title'].replace(['Mlle','Ms','Lady'],'Miss')
train_data['title']=train_data['title'].replace('Sir','Mr')
train_data['title']=train_data['title'].replace('Mme','Mrs')
print(train_data['title'].value_counts())

title
Mr        518
Miss      186
Mrs       126
Master     40
rare       21
Name: count, dtype: int64


In [32]:
title_mapping={"Mr":1,"Miss":2,"Mrs":3,"Master":4,"rare":5}
train_data['title']=train_data['title'].map(title_mapping)
print(train_data['title'].value_counts())

title
1    518
2    186
3    126
4     40
5     21
Name: count, dtype: int64


In [33]:
#处理出发地缺失数据（个人认为直接用最多的替换比较合适）
print(train_data['Embarked'].value_counts())
train_data['Embarked']=train_data['Embarked'].fillna('S')
print(train_data['Embarked'].value_counts())
train_data['Embarked']=train_data['Embarked'].map({'S':0,'C':1,'Q':2})
print(train_data['Embarked'].value_counts())

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64
Embarked
S    646
C    168
Q     77
Name: count, dtype: int64
Embarked
0    646
1    168
2     77
Name: count, dtype: int64


In [34]:
##第一次评估结果不够好，引入交叉验证
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
# 5折交叉验证
scores = cross_val_score(model, x_train, y_train, cv=5)
print(f"交叉验证平均准确率: {scores.mean():.4f} (+/- {scores.std():.4f})")

交叉验证平均准确率: 0.8342 (+/- 0.0196)


In [35]:
##引入网格搜索
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# 定义暴力搜索的参数范围
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 5, 6, 7],
    'min_samples_split': [2, 5, 10]
}

# 开始搜索
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
grid_search.fit(x_train, y_train)

print(f"最佳参数: {grid_search.best_params_}")
print(f"最佳交叉验证分数: {grid_search.best_score_:.4f}")


best_rf_model = grid_search.best_estimator_

最佳参数: {'max_depth': 6, 'min_samples_split': 5, 'n_estimators': 100}
最佳交叉验证分数: 0.8384


In [36]:
#训练和评估
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
x_train,x_val,y_train,y_val=train_test_split(x,y,test_size=0.2,random_state=42)
model=RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42)
model.fit(x_train,y_train)
y_pred=model.predict(x_val)
print(f"本地准确性:{accuracy_score(y_val,y_pred):.4f}")

本地准确性:0.8268


In [37]:
#逻辑回归
from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(x_train, y_train)
lr_pred = lr_model.predict(x_val)
lr_acc = accuracy_score(y_val, lr_pred)
print(f"逻辑回归 本地验证准确率: {lr_acc:.4f}")
print(f"\n模型对比：")
print(f"随机森林: {accuracy_score(y_val, model.predict(x_val)):.4f}")
print(f"逻辑回归: {lr_acc:.4f}")

逻辑回归 本地验证准确率: 0.7765

模型对比：
随机森林: 0.8268
逻辑回归: 0.7765


In [38]:
print(x.isnull().sum())

Pclass         0
Sex            0
Age            0
Fare           0
Embarked       0
title          0
Family_size    0
dtype: int64


In [11]:
#预备生成后提交
#  检查当前内存里的 train_data 的状态
print("当前 train_data 的列：")
print(train_data.columns)
print("当前 train_data 里 Sex 的值：")
print(train_data['Sex'].unique())  

# 清洗test
# 处理 Title
test_data['title'] = test_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test_data['title'] = test_data['title'].replace(['Countess','Capt','Col','Don','Dr','Major','Rev','Jonkheer','Dona'],'rare')
test_data['title'] = test_data['title'].replace(['Mlle','Ms','Lady'],'Miss')
test_data['title'] = test_data['title'].replace('Sir','Mr')
test_data['title'] = test_data['title'].replace('Mme','Mrs')
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "rare": 5}
test_data['title'] = test_data['title'].map(title_mapping)

# 处理 Sex 和 Embarked
test_data['Sex'] = test_data['Sex'].map({'male': 1, 'female': 2})
test_data['Embarked'] = test_data['Embarked'].fillna('S')
test_data['Embarked'] = test_data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# 构造 Family_size
test_data['Family_size'] = test_data['SibSp'] + test_data['Parch'] + 1

# 处理 Age 和 Fare 缺失值
train_age_median = train_data['Age'].median() if not train_data['Age'].isnull().any() else 28.0 #28.0来源于历史数据
train_fare_median = train_data['Fare'].median() if not train_data['Fare'].isnull().any() else 14.45

test_data['Age'] = test_data['Age'].fillna(train_age_median)
test_data['Fare'] = test_data['Fare'].fillna(train_fare_median)

# 挑出测试集的特征
features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'title', 'Family_size']
X_test_final = test_data[features]
predictions = model.predict(X_test_final)

#  生成 Kaggle 提交文件
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': predictionsb
})
submission.to_csv('submission.csv', index=False)

print(" submission.csv 已生成！")
print(submission.head())

当前 train_data 的列：
Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'title', 'Family_size'],
      dtype='str')
当前 train_data 里 Sex 的值：
[1 2]
 submission.csv 已生成！
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1


In [43]:
import pandas as pd
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression

# ==========================================
#  重新加载原始测试集（防止内存丢失导致的状态混乱）
# ==========================================
test_data = pd.read_csv('test.csv')

# ==========================================
#  对测试集进行和训练集一模一样的特征工程
# ==========================================
# 提取 title
test_data['title'] = test_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test_data['title'] = test_data['title'].replace(['Countess','Capt','Col','Don','Dr','Major','Rev','Jonkheer','Dona'],'rare')
test_data['title'] = test_data['title'].replace(['Mlle','Ms','Lady'],'Miss')
test_data['title'] = test_data['title'].replace('Sir','Mr')
test_data['title'] = test_data['title'].replace('Mme','Mrs')
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "rare": 5}
test_data['title'] = test_data['title'].map(title_mapping)

# 处理 Sex 和 Embarked
test_data['Sex'] = test_data['Sex'].map({'male': 1, 'female': 2})
test_data['Embarked'] = test_data['Embarked'].fillna('S')
test_data['Embarked'] = test_data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# 构造家庭大小（注意这里是大写 F：Family_size）
test_data['Family_size'] = test_data['SibSp'] + test_data['Parch'] + 1

# 填补缺失值（用训练集的中位数！确保 train_data 还在内存中）
test_data['Age'] = test_data['Age'].fillna(train_data['Age'].median())
test_data['Fare'] = test_data['Fare'].fillna(train_data['Fare'].median())

# ==========================================
#  定义测试集特征（注意大小写保持一致）
# ==========================================
features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'title', 'Family_size']
x_test = test_data[features]

# 检查一下是否还有缺失值
print("测试集缺失值情况：")
print(x_test.isnull().sum())

# ==========================================
#  使用综合模型预测（假设 x_train, y_train, best_rf_model 还在内存里）
# ==========================================
best_rf = best_rf_model
lr = LogisticRegression(max_iter=1000, random_state=42)

voting_clf = VotingClassifier(
    estimators=[('lr', lr), ('best_rf', best_rf)],
    voting='soft'
)

# 训练投票委员会
voting_clf.fit(x_train, y_train)

# 预测
voting_pred = voting_clf.predict(x_test)

# ==========================================
# 5. 生成提交文件
# ==========================================
submission_voting = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': voting_pred
})
submission_voting.to_csv('submission_voting_tuned.csv', index=False)

print("\n 综合了【网格搜索】和【集成投票】的提交文件已生成！")
print("文件名: submission_voting_tuned.csv")
print(submission_voting.head())

测试集缺失值情况：
Pclass         0
Sex            0
Age            0
Fare           0
Embarked       0
title          0
Family_size    0
dtype: int64

🎉 综合了【网格搜索】和【集成投票】的提交文件已生成！
文件名: submission_voting_tuned.csv
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1
